# 06 · Test the best model: image in → JSON out

Loads `checkpoints/best.pt`, lets you point at **any table image** (an
upload/copy of your own, or a held-out validation render), and shows the
image, the raw generation, and the parsed/repaired JSON side by side.

In [ ]:
# --- bootstrap: make the src/ package importable from notebooks/ ---
import sys, os
from pathlib import Path
REPO = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)  # so relative paths in configs/config.yaml resolve

from gemma_ft_json.config import load_config
cfg = load_config("configs/config.yaml")
print("config loaded; data_root =", cfg.paths.data_root)


## 1. Rebuild the model and load `best.pt`

In [ ]:
import torch
from gemma_ft_json.models import load_gemma_local, GemmaVisionForJSON
from gemma_ft_json.utils import resolve_device, resolve_dtype

device = resolve_device(cfg.device.preferred, cfg.device.mps_fallback_env)
gemma, tok = load_gemma_local(cfg.paths.gemma_model_dir, resolve_dtype(cfg.device.dtype))
model = GemmaVisionForJSON(gemma, tok, cfg.model,
                           image_size=cfg.dataset.image_size, stage="sft")

ckpt_path = str(REPO / "checkpoints" / "best.pt")
ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
model.load_trainable_state_dict(ck["model"])
print(f"loaded {ckpt_path} (epoch {ck['epoch']+1}, best_val {ck['best_val']:.4f})")

## 2. Pick an image
Set `IMAGE_PATH` to **your own image** (drop a file into the repo and point at
it), or leave `None` to grab a held-out validation sample (with ground truth
for comparison).

In [ ]:
import json
from PIL import Image

IMAGE_PATH = None     # e.g. "my_table_photo.png"
gt = None
if IMAGE_PATH is None:
    rec = json.loads(open(cfg.paths.manifest_val).readline())
    IMAGE_PATH, gt = rec["image"], json.loads(rec["json"])
img = Image.open(IMAGE_PATH).convert("RGB")
print("testing:", IMAGE_PATH)

## 3. Predict

In [ ]:
from gemma_ft_json.inference import Predictor
pred = Predictor(model, cfg, device)
out = pred.predict(img)            # greedy + JSON bracket-balance guard
print("strictly valid JSON:", out["strictly_valid"])
print("--- raw generation (first 400 chars) ---")
print(out["raw_text"][:400])

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(img); axes[0].axis("off"); axes[0].set_title("input image")
axes[1].axis("off")
axes[1].text(0, 1, json.dumps(out["json"], indent=2)[:1500] if out["json"]
             else "(unparseable)", fontsize=8, family="monospace", va="top")
axes[1].set_title("predicted JSON" + ("" if out["strictly_valid"] else " (repaired)"))
plt.tight_layout(); plt.show()

## 4. Cell-level accuracy vs ground truth (validation samples only)

In [ ]:
def cell_accuracy(pred_obj, gt_obj):
    if not pred_obj or "rows" not in pred_obj:
        return 0.0
    hits = tot = 0
    for gr, pr in zip(gt_obj["rows"], pred_obj.get("rows", [])):
        for gc, pc in zip(gr, pr):
            tot += 1; hits += (str(gc).strip() == str(pc).strip())
    tot += abs(len(gt_obj["rows"]) - len(pred_obj.get("rows", []))) * len(gt_obj["columns"])
    return hits / max(1, tot)

if gt is not None:
    print(f"cell accuracy: {cell_accuracy(out['json'], gt):.1%}")
    print("GT columns  :", gt["columns"])
    print("pred columns:", (out["json"] or {}).get("columns"))
else:
    print("no ground truth for custom images — inspect visually above")